# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url)

print(df.shape)
print(df.isnull().sum())

(9404, 32)
hotel                                0
is_canceled                          0
lead_time                            0
arrival_date_year                    0
arrival_date_month                   0
arrival_date_week_number             0
arrival_date_day_of_month            0
stays_in_weekend_nights              0
stays_in_week_nights                 0
adults                               0
children                             0
babies                               0
meal                                 0
country                            266
market_segment                       0
distribution_channel                 0
is_repeated_guest                    0
previous_cancellations               0
previous_bookings_not_canceled       0
reserved_room_type                   0
assigned_room_type                   0
booking_changes                      0
deposit_type                         0
agent                             2335
company                           8393
days_in_waitin

In [4]:
# Use 10,000 rows for faster modeling
df = df.sample(n=min(10000, len(df)), random_state=42)

X = df.drop(columns=['is_canceled', 'reservation_status',
                     'reservation_status_date', 'assigned_room_type'])
y = df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [5]:
# Fill missing values and encode categories
X_train = pd.get_dummies(X_train, dummy_na=True)
X_test = pd.get_dummies(X_test, dummy_na=True)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

### ✍️ Your Response: 🔧
1. In the dataset, there are 9,404 rows and 32 columns.

2. The dataset includes categorical like hotel type, meal type, country, deposit type, etc and numerical like adults, children, previous cancellations, etc.

3. Some steps I took to clean and prepare the data was I checked to see if there were any missing values, I also encoded categorical varibales using one-hot encoding, and I split the data into training and test sets (70/30).

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [12]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb = GaussianNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))
print(confusion_matrix(y_test, nb_pred))

Accuracy: 0.5194897236002834
              precision    recall  f1-score   support

           0       1.00      0.36      0.53      2121
           1       0.34      1.00      0.51       701

    accuracy                           0.52      2822
   macro avg       0.67      0.68      0.52      2822
weighted avg       0.83      0.52      0.53      2822

[[ 768 1353]
 [   3  698]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. It says that my model performed 51.95% of accuracy which is okay. I think the F1 score is the best metric used to judge the performance.

2. This model might be useful for the hotel to find bookings that are more likely to be canceled and let the customers know. But it can be improved a bit more before using.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel='linear')
svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))
print(confusion_matrix(y_test, svm_pred))


Accuracy: 0.9110559886605244
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      2121
           1       0.84      0.80      0.82       701

    accuracy                           0.91      2822
   macro avg       0.89      0.87      0.88      2822
weighted avg       0.91      0.91      0.91      2822

[[2011  110]
 [ 141  560]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. It says that my model performed 91.11% of accuracy which is really good. I think the F1 score is still the best metric used to judge the performance.

2. SVM could provide better insights than simpler models because it can help predict cancellations based on a lot of factors, provide more accuracy, without a lot of false predictions.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [10]:
from sklearn.neural_network import MLPClassifier

X_fit, X_val, y_fit, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

nn = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=200, random_state=42)

nn.fit(X_fit, y_fit)

print("Validation Accuracy:", nn.score(X_val, y_val))

nn_pred = nn.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, nn_pred))
print(classification_report(y_test, nn_pred))
print(confusion_matrix(y_test, nn_pred))


Validation Accuracy: 0.8967350037965072
Test Accuracy: 0.9050318922749823
              precision    recall  f1-score   support

           0       0.94      0.93      0.94      2121
           1       0.81      0.81      0.81       701

    accuracy                           0.91      2822
   macro avg       0.87      0.87      0.87      2822
weighted avg       0.91      0.91      0.91      2822

[[1983  138]
 [ 130  571]]


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. It says that this modeled had an accuracy of 90.50% which is better than the Naive Bayes, but not as good as the SMV one.

2. I think the business might not be comfortable with it, as it's more harder to understand how it makes its predictions because it's more complicated than the other models. We can see what goes in and what comes out, but it's harder to understand.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [14]:
print("Naive Bayes:", accuracy_score(y_test, nb_pred))
print("SVM:", accuracy_score(y_test, svm_pred))
print("Neural Network:", accuracy_score(y_test, nn_pred))

Naive Bayes: 0.5194897236002834
SVM: 0.9110559886605244
Neural Network: 0.9050318922749823


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. After conducting all of the models, I found out that SVM had the best overall accuracy of 91.11%. But I think that Naive Bayes had the best training time, interpretability, and it's easiest to use.

2. I would still recommend SVM though for deployment because it had the highest accuracy and F1 score, and I think I would be great in its performance espcially with prediction to make the best decisions.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I would recommend implementing the SVM model because it will help the hotel management predict cancellations the most accurate which will help you make better decisions in the future. I would say not to fully rely on just the model itself, and I would recommend adding other data like booking history to improve the results in the future.

2. This relates to my customized learning outcome because I was able to build and evaluate models to help make better financial decisions. I was able to build models, compare the accuracy, and see how it could apply in real life.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"